**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/mehrotra-2024"], check=True)
    os.chdir("/content/encore/entries/mehrotra-2024")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Mehrotra, Levenstein et al. (2024): the Isomap ring claims, on dandiset 000939

Paper: Mehrotra, Levenstein et al. (2024), Current Biology 34, 3043-3054,
[doi:10.1016/j.cub.2024.05.048](https://doi.org/10.1016/j.cub.2024.05.048). The registry's target is
dandiset 000987 (`reproduce_mehrotra_2024.ipynb`); these claims use the paper's PoSub data, deposited
as DANDI:000939, and are counted apart (`data: elsewhere`).

Dataset: [DANDI:000939](https://dandiarchive.org/dandiset/000939/0.261016.2310), version
`0.261016.2310` (the most recent published version; the paper cites none; see `posub_000939.ipynb`
for the choice and the 16 sessions, taken from the event files of the paper's code at commit
4ee34f3). Units, epochs, position and head direction are streamed.

Claims file: `claims/mehrotra-2024.yaml`. The ring follows makeRing.py of the paper's code and the
STAR Methods (Isomap Projection and Decoding), as `mehrotra.ring_session` documents. The paper has
no code for Figures 1J and 1K; their measures follow the Methods and the claims file.

| Claim | Here |
|---|---|
| fig1j-down-resets-hd | tested |
| fig1k-hd-stable-at-onset | tested |

The other PoSub claims are in `posub_000939.ipynb` and `posub_lfp_000939.ipynb`.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pynapple as nap
from scipy.stats import mannwhitneyu
import mehrotra as m

print("pynapple", nap.__version__, "| dandiset", m.POSUB_DANDISET, m.POSUB_VERSION)
S = m.load_all_posub()
assets = {a["path"].split("/")[0].replace("sub-", ""): a for a in m.list_assets(m.POSUB_DANDISET, m.POSUB_VERSION)}
# DOWN and UP states as in posub_000939.ipynb: the paper's event files, except A3708-200317b, whose
# event file does not share the clock of the spikes; its states are re-detected from the spikes.
EV = {}
for s in S:
    if s["session"] == "A3708-200317b":
        EV[s["session"]] = m.detect_down_states(s["spikes"], s["nrem_hc"], restrict_up=True)
    else:
        EV[s["session"]] = (s["down"], s["up"])
print(len(S), "sessions")

## Data check and the ring of each session

For each session, the ring is fitted and the NREM steps around up to 600 long UP states (longer than
0.5 s, as makeRing.py and the Methods select) are projected. As a check that the ring encodes head
direction, the angle of each wake bin on the ring is compared with the animal's head direction in
that bin: the mean resultant length of their difference (or of their sum, for a mirrored ring) is 1
for a perfect ring and near 0 for none. The radius threshold is one third of the mean wake radius;
the median lag of the first threshold crossing after DU is given against the 35 ms the Methods name
as the mode.
Isomap is run with the dense eigen solver, so that the ring is the same in every run (the default
ARPACK solver starts from an unseeded random vector, so the ring can come out rotated or reflected
differently from run to run), and each ring is oriented so that its angle increases with head
direction in wake (`mirrored`), since the solver's reflection is arbitrary and a reflection negates
every angle difference of Figure 1J.


In [ ]:
RING = {}
rows = []
for k, s in enumerate(S):
    t0 = time.time()
    rng = np.random.default_rng(1000 + k)
    e = s["epochs"][s["epochs"].tag.str.startswith("wake")]
    wake = nap.IntervalSet(start=e.start.values, end=e.end.values)
    down, up = EV[s["session"]]
    hdu = s["units"].index[s["units"].hd].tolist()
    r = m.ring_session(assets[s["session"].split("-")[0]], s["spikes"], hdu, wake, up, down, rng)
    RING[s["session"]] = r
    thr = r["wake_mean_radius"] / 3
    post = (r["lags"] >= 0) & (r["lags"] < 0.15)
    first = [r["lags"][post][np.argmax(x)] for x in (r["radius"][:, post] > thr) if x.any()]
    rows.append(dict(session=s["session"], hd_cells=r["n_hd"], wake_bins_moving=r["n_wake_bins"], fit_bins=r["n_fit"],
                     long_up=r["n_long_up"], projected_up=len(r["du"]), with_down_before=int(np.isfinite(r["down_dur"]).sum()),
                     wake_mean_radius=round(r["wake_mean_radius"], 2), ring_angle_vs_hd_R=round(r["wake_angle_vs_hd"], 3), mirrored=r["mirrored"],
                     sleep_radius_q10_50_90=tuple(np.round(r["sleep_radius_quantiles"], 2)),
                     first_crossing_median_ms=round(1000 * float(np.median(first)), 1) if first else np.nan,
                     seconds=round(time.time() - t0)))
RT = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print(RT.to_string())
print("ring angle against HD, R from", RT.ring_angle_vs_hd_R.min(), "to", RT.ring_angle_vs_hd_R.max(),
      "| rings reflected to follow head direction:", int(RT.mirrored.sum()), "of", len(RT))

In [ ]:
# fig1j and fig1k from the ring of each session (RING: session -> mehrotra.ring_session output).
def circ_mean(a):
    return np.angle(np.mean(np.exp(1j * a))) if len(a) else np.nan

def angles_at_transitions(r):
    """Per projected UP state: thr-passing angle at UD (two readings) and the first post-DU angle."""
    thr = r["wake_mean_radius"] / 3
    lags = r["lags"]; rad = r["radius"]; ang = r["angle"]
    out = []
    for i in range(len(r["du"])):
        dd = r["down_dur"][i]
        if not np.isfinite(dd):
            continue
        above = rad[i] > thr
        pre = (lags < -dd) & above
        post = np.where((lags >= 0) & (lags < 0.15) & above)[0]
        if not pre.any() or len(post) == 0:
            continue
        last = np.where(pre)[0][-1]
        win = pre & (lags >= -dd - 0.15)
        out.append(dict(ud_last=ang[i, last], ud_mean150=circ_mean(ang[i, win]), du_first=ang[i, post[0]],
                        du_first_lag=lags[post[0]]))
    return pd.DataFrame(out)

def wrap(x):
    return np.angle(np.exp(1j * x))

def cumvar_slope(r, t0, t1, kind, mask=True, per_up=False):
    """Session diffusion coefficient: the cumulative circular variance of the thr-passing angles from
    t0 to t1 after DU, from the first three bins, averaged over UP states; mean numpy gradient over
    t0 + 35 ms to t1. kind 'scipy' is 1 - R (scipy.stats.circvar since 1.8), 'log' is -2 ln R.
    mask=False keeps the steps below the radius threshold; per_up=True takes the slope of each UP
    state's curve and averages the slopes."""
    thr = r["wake_mean_radius"] / 3
    lags = r["lags"]; w = (lags >= t0 - 1e-9) & (lags < t1 - 1e-9)
    tl = lags[w]
    curves = []
    for i in range(len(r["du"])):
        if not (r["radius"][i, w] > thr).any():
            continue
        a = np.where(r["radius"][i, w] > thr, r["angle"][i, w], np.nan) if mask else r["angle"][i, w]
        z = np.exp(1j * a)
        ok = np.isfinite(a)
        if ok.sum() < 3:
            continue
        cs = np.cumsum(np.where(ok, z, 0)); cn = np.cumsum(ok)
        R = np.abs(cs) / np.maximum(cn, 1)
        cv = (1 - R) if kind == "scipy" else -2 * np.log(np.maximum(R, 1e-12))
        cv[cn < 3] = np.nan
        cv[:2] = np.nan
        curves.append(cv)
    sel = tl >= t0 + 0.035 - 1e-9
    if per_up:
        sl = []
        for cv in curves:
            g = np.gradient(cv, tl)
            if np.isfinite(g[sel]).any():
                sl.append(np.nanmean(g[sel]))
        return float(np.mean(sl)), len(sl)
    with np.errstate(invalid="ignore"):
        curve = np.nanmean(np.array(curves), 0)
    g = np.gradient(curve, tl)
    return float(np.nanmean(g[sel])), len(curves)

def fig1j_test(TR, rng, col="ud_last", nbins=20, n_shuffle=1000):
    """Pooled histogram of the angle after DU minus the angle at UD, against shuffles of the angle at UD
    within each session: bins outside the shuffles' minimum to maximum, resultant length, and the
    shuffle counts."""
    obs = np.concatenate([np.asarray(wrap(v.du_first.values - v[col].values)) for v in TR.values()])
    obs = obs[np.isfinite(obs)]
    shuf = []
    for _ in range(n_shuffle):
        sh = []
        for v in TR.values():
            u = v[col].values
            ok = np.isfinite(u)
            sh.append(wrap(v.du_first.values[ok] - rng.permutation(u[ok])))
        shuf.append(np.concatenate(sh))
    shuf = np.array(shuf)
    edges = np.linspace(-np.pi, np.pi, nbins + 1)
    h = np.histogram(obs, edges)[0]
    hs = np.array([np.histogram(x, edges)[0] for x in shuf])
    lo, hi = hs.min(0), hs.max(0)
    return dict(n=len(obs), bins_outside=int(((h < lo) | (h > hi)).sum()), h=h, lo=lo, hi=hi, hs=hs, edges=edges,
                resultant_length=float(np.abs(np.mean(np.exp(1j * obs)))),
                shuffle_R_99=float(np.quantile(np.abs(np.mean(np.exp(1j * shuf), 1)), 0.99)))


## Claim fig1j-down-resets-hd

> The DOWN state resets the virtual HD signal; the angle at the start of an UP state is unrelated to
> the angle at the end of the previous UP state.

Criterion: angles from the Isomap ring; the first post-DU step whose radius exceeds the threshold
(searched 0 to 150 ms after DU) against the angle at the preceding UD transition, read as the last
above-threshold step before UD, or as the circular mean of the above-threshold steps in the last
150 ms before UD. UD is the start of the DOWN state that ends at the UP state's start. Only UP states
with an above-threshold step on both sides are used (Methods). The histogram of angular differences,
pooled over sessions, is compared with 1,000 shuffles of the angle at UD within each session;
reproduced when no bin lies outside the minimum to maximum of the shuffled counts. 20 bins is the
standard choice; 10 and 36 are shown.

In [ ]:
rng_j = np.random.default_rng(20240601)
TR = {k: angles_at_transitions(r) for k, r in RING.items()}
print("UP states used per session:", {k: len(v) for k, v in TR.items()})
res_j = []
for col in ("ud_last", "ud_mean150"):
    for nb in (20, 10, 36):
        t = fig1j_test(TR, rng_j, col, nb)
        res_j.append(dict(ud_angle=col, bins=nb, n=t["n"], bins_outside=t["bins_outside"], reproduced=t["bins_outside"] == 0,
                          obs_min=int(t["h"].min()), obs_max=int(t["h"].max()), shuffle_min=int(t["lo"].min()),
                          shuffle_max=int(t["hi"].max()), resultant_length=round(t["resultant_length"], 4),
                          shuffle_R_99=round(t["shuffle_R_99"], 4)))
        if nb == 20 and col == "ud_last":
            H20 = (t["edges"], t["h"], t["lo"], t["hi"])
            HS20 = t["hs"]
RJ = pd.DataFrame(res_j)
print(RJ.to_string())
# Chance rate of the test: each shuffle taken as the data, against the range of the other 999.
fp = np.mean([((HS20[i] < np.delete(HS20, i, 0).min(0)) | (HS20[i] > np.delete(HS20, i, 0).max(0))).any() for i in range(len(HS20))])
print(f"20 bins, last angle before UD: fraction of shuffles with a bin outside the range of the others: {fp:.3f}")
edges, h, lo, hi = H20
cen = (edges[:-1] + edges[1:]) / 2
outb = np.where((h < lo) | (h > hi))[0]
print("20 bins, last angle before UD: bins outside the shuffle range:",
      [(round(float(cen[b]), 2), int(h[b]), int(lo[b]), int(hi[b])) for b in outb], "(centre rad, count, shuffle min, max)")
print("count in the bin at 0 difference:", int(h[np.argmin(np.abs(cen))]), "| shuffle range there:",
      int(lo[np.argmin(np.abs(cen))]), "to", int(hi[np.argmin(np.abs(cen))]))
# Variant: only sessions whose ring follows head direction in wake (R >= 0.5).
good = RT.session[RT.ring_angle_vs_hd_R >= 0.5].tolist()
tg = fig1j_test({k: TR[k] for k in good}, rng_j)
print(f"variant, {len(good)} sessions with ring R >= 0.5, last angle, 20 bins: bins outside {tg['bins_outside']}, "
      f"resultant length {tg['resultant_length']:.4f}")
# Variant: each ring as the eigen solver returns it, without the orientation to head direction
# (the reflected sessions have their angles negated back).
TR_raw = {k: (v.assign(ud_last=-v.ud_last, ud_mean150=-v.ud_mean150, du_first=-v.du_first) if RING[k]["mirrored"] else v)
          for k, v in TR.items()}
for col in ("ud_last", "ud_mean150"):
    tr_ = fig1j_test(TR_raw, rng_j, col)
    print(f"variant, rings as the solver returns them, {col}, 20 bins: bins outside {tr_['bins_outside']}, "
          f"resultant length {tr_['resultant_length']:.4f}")
per = []
for k, v in TR.items():
    d = wrap(v.du_first.values - v.ud_last.values)
    per.append(dict(session=k, n=len(v), resultant_length=round(float(np.abs(np.mean(np.exp(1j * d)))), 3)))
print(pd.DataFrame(per).to_string())
c = cen
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.fill_between(c, lo, hi, step="mid", alpha=0.3, label="shuffle min to max")
ax.step(c, h, where="mid", color="k", label="data")
ax.set_xlabel("angle after DU minus angle at UD (rad)"); ax.set_ylabel("UP states"); ax.legend()
plt.tight_layout(); plt.show()

**Result: not reproduced.** Over 6,510 UP states of 16 sessions, the angle at the first
above-threshold step after DU is nearly unrelated to the last above-threshold angle before UD (mean
resultant length of the differences 0.020, against 0.078 for the 99th percentile of the shuffles),
but under the criterion's test one of 20 bins lies outside the minimum to maximum of the 1,000
shuffles: a deficit at -0.47 rad (294 UP states, shuffle range 300 to 422); the bin at zero
difference is inside its range (409, against 368 to 491). With 10 bins one bin lies outside and
with 36 two; with the mean angle of the last 150 ms before UD (5,043 UP states) two bins with 20
bins, one with 10 and none with 36. With this procedure a shuffle lies outside the range of the
others in at least one of 20 bins 3.3% of the time. On the 14 sessions whose ring follows head
direction in wake (R >= 0.5; A3706-200313 has 0.114 and A3730-210323b 0.328), two bins lie outside
(resultant length 0.032). With the rings as the eigen solver returns them (7 of 16 are reflected to
follow head direction), one bin lies outside under both angles at UD. The outcome is fragile: with
the rings fitted on other random draws (below), one bin lies outside with seeds 3000 + k and none
with seeds 2000 + k, where the claim would be reproduced. The criterion decides the outcome; the
absence of a relation holds in size, and the test fails by one bin in two of three draws.


## Claim fig1k-hd-stable-at-onset

> During the UP onset sequence the represented HD changes less rapidly than later in the UP state
> (diffusion coefficients, Mann-Whitney U test, U = 54, n = 16, p = 0.006).

Criterion: UP states longer than 500 ms; angles 0 to 150 ms (traveling) and 150 to 300 ms
(control) after DU; cumulative circular variance from the first three bins; the diffusion
coefficient is the mean numpy gradient of it between 35 and 150 ms (for the control interval, the
same offsets, 185 to 300 ms). Per session, one coefficient per interval: the cumulative variance is
averaged over the session's UP states, then its slope taken. Steps below the radius threshold are
left out, as the Methods compute angles only above it. Reproduced when traveling is lower than
control by a two-sided Mann-Whitney U test at p < 0.05.

The Methods use SciPy's `circvar`, which since SciPy 1.8 is 1 - R (R the mean resultant length) and
before was 2 ln(1/R); the paper does not give the version, so both are run.

In [ ]:
res_k = []
for kind in ("scipy", "log"):
    tr = {k: cumvar_slope(r, 0.0, 0.15, kind) for k, r in RING.items()}
    ct = {k: cumvar_slope(r, 0.15, 0.30, kind) for k, r in RING.items()}
    a = np.array([v[0] for v in tr.values()]); b = np.array([v[0] for v in ct.values()])
    mw = mannwhitneyu(a, b)
    res_k.append(dict(circvar=kind, sessions=len(a), traveling_median=round(float(np.median(a)), 3),
                      control_median=round(float(np.median(b)), 3), traveling_lower_sessions=int((a < b).sum()),
                      U=mw.statistic, p=mw.pvalue, reproduced=bool(np.median(a) < np.median(b) and mw.pvalue < 0.05)))
    if kind == "scipy":
        K = pd.DataFrame(dict(traveling=a, control=b, up_traveling=[v[1] for v in tr.values()],
                              up_control=[v[1] for v in ct.values()]), index=list(RING))
good = RT.session[RT.ring_angle_vs_hd_R >= 0.5].tolist()
for label, kw, names in (("steps below threshold kept", dict(mask=False), list(RING)),
                         ("slope per UP state, averaged", dict(per_up=True), list(RING)),
                         ("sessions with ring R >= 0.5", {}, good)):
    a = np.array([cumvar_slope(RING[k], 0.0, 0.15, "scipy", **kw)[0] for k in names])
    b = np.array([cumvar_slope(RING[k], 0.15, 0.30, "scipy", **kw)[0] for k in names])
    mw = mannwhitneyu(a, b)
    res_k.append(dict(circvar="scipy, " + label, sessions=len(a), traveling_median=round(float(np.median(a)), 3),
                      control_median=round(float(np.median(b)), 3), traveling_lower_sessions=int((a < b).sum()),
                      U=mw.statistic, p=mw.pvalue, reproduced=bool(np.median(a) < np.median(b) and mw.pvalue < 0.05)))
print(pd.DataFrame(res_k).to_string())
print("\npaper: U = 54, n = 16, p = 0.006")
print(K.round(3).to_string())

**Result: not reproduced, resting on an assumption.** With the steps below the radius threshold
left out, as the Methods compute angles only above it, the diffusion coefficient during the
traveling UP state is lower than in the control interval in 10 of 16 sessions (median 0.373 against
0.439 per s), and the two-sided Mann-Whitney test does not find a difference (U = 98, p = 0.27;
paper U = 54, p = 0.006). With the older circular variance, 2 ln(1/R), it is lower in 12 of 16
(U = 87, p = 0.13); with the slope of each UP state's curve averaged, in 6 of 16 (p = 0.25); on the
14 sessions whose ring follows head direction, in 8 of 14 (U = 83, p = 0.51). Keeping the steps
below the threshold gives 12 of 16 (U = 76, p = 0.052) in this draw, and p = 0.025 and 0.044 with
the rings of the other two draws (below), near the paper's result. With the steps left out, the
other two draws give p = 0.10 and 0.21, so the outcome does not depend on the draw. It rests on
leaving those steps out, which the Methods support ("one-third of the mean wake radius ... which we
used as a threshold for computing angular direction"; the slope starts at 35 ms because the
threshold crossing varies between UP states).


## Repeatability: the ring again, and with other draws

The ring rests on random draws: the half of the wake bins and the 1% of NREM bins it is fitted on,
and the long UP states projected. First, the ring of the first session is fitted a second time with
the same generator, to show that a run gives the same angles. Then the rings of all sessions are
fitted again with two other generators per session (seeds 2000 + k and 3000 + k in place of
1000 + k), and the main tests of fig1j and fig1k are repeated on them.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def ring_all(seed):
    def one(k):
        s = S[k]
        e = s["epochs"][s["epochs"].tag.str.startswith("wake")]
        wake = nap.IntervalSet(start=e.start.values, end=e.end.values)
        down, up = EV[s["session"]]
        hdu = s["units"].index[s["units"].hd].tolist()
        return s["session"], m.ring_session(assets[s["session"].split("-")[0]], s["spikes"], hdu, wake, up, down,
                                            np.random.default_rng(seed + k))
    with ThreadPoolExecutor(3) as ex:
        return dict(ex.map(one, range(len(S))))

t0 = time.time()
k0 = S[0]["session"]
s0 = S[0]; e0 = s0["epochs"][s0["epochs"].tag.str.startswith("wake")]
again = m.ring_session(assets[k0.split("-")[0]], s0["spikes"], s0["units"].index[s0["units"].hd].tolist(),
                       nap.IntervalSet(start=e0.start.values, end=e0.end.values), *EV[k0][::-1], np.random.default_rng(1000))
print(f"{k0} fitted again with the same generator: largest difference in radius {np.abs(again['radius'] - RING[k0]['radius']).max():.2e}, "
      f"in angle {np.abs(wrap(again['angle'] - RING[k0]['angle'])).max():.2e} rad")

rows_s = []
for seed in (1000, 2000, 3000):
    R_ = RING if seed == 1000 else ring_all(seed)
    TR_ = {k: angles_at_transitions(r) for k, r in R_.items()}
    rng_s = np.random.default_rng(20240601)
    t_last, t_mean = fig1j_test(TR_, rng_s, "ud_last"), fig1j_test(TR_, rng_s, "ud_mean150")
    row = dict(seed=f"{seed} + k", ring_R_min=round(min(r["wake_angle_vs_hd"] for r in R_.values()), 3),
               j_up_states=t_last["n"], j_last_bins_outside=t_last["bins_outside"], j_last_R=round(t_last["resultant_length"], 4),
               j_last_shuffle_R_99=round(t_last["shuffle_R_99"], 4), j_mean150_bins_outside=t_mean["bins_outside"])
    for label, kw in (("k", {}), ("k_kept", dict(mask=False))):
        a = np.array([cumvar_slope(R_[k], 0.0, 0.15, "scipy", **kw)[0] for k in R_])
        b = np.array([cumvar_slope(R_[k], 0.15, 0.30, "scipy", **kw)[0] for k in R_])
        mw = mannwhitneyu(a, b)
        row.update({f"{label}_lower": int((a < b).sum()), f"{label}_U": mw.statistic, f"{label}_p": round(mw.pvalue, 4)})
    rows_s.append(row)
print(pd.DataFrame(rows_s).to_string())
print(f"({time.time() - t0:.0f} s)")


**Result: a run repeats, and the outcomes hold in two of three draws for fig1j and in all three
for fig1k.** Fitted again with the same generator, the ring of A3701-191119 is identical (largest
difference 0 in radius and angle), so a run of this notebook repeats. With the rings fitted on other
draws, fig1j has one bin outside the shuffle range with seeds 1000 + k (the main draw) and 3000 + k
and none with 2000 + k (resultant length 0.020 to 0.030, against shuffle 99th percentiles of 0.070
to 0.078); fig1k, with steps below the threshold left out, gives p = 0.27, 0.10 and 0.21 (traveling
lower in 10, 14 and 10 of 16 sessions), and with them kept p = 0.052, 0.025 and 0.044.


## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1j-down-resets-hd | not reproduced (one of 20 bins outside the shuffle range; resultant length 0.020; none in one of three draws) |
| fig1k-hd-stable-at-onset | not reproduced, resting on an assumption (U = 98, p = 0.27) |

How the analysis differs from the paper's. The ring follows makeRing.py, with the file's HD flag for
HD cells, a cap of 2,500 wake bins and 600 long UP states per session for run time, the dense eigen
solver so that a run repeats, each ring oriented to follow head direction, and the event
files of the paper's code for UP and DOWN states (A3708-200317b re-detected, as in
`posub_000939.ipynb`). The paper has no code for Figures 1J and 1K: the angle at UD, the pooling of
sessions and the shuffle within sessions for 1J, and the handling of steps below the threshold, the
averaging over UP states and the circular variance for 1K are choices shown above with their
alternatives.

What was found about the dataset (numbers above): the wake ring follows head direction in 14 of 16
sessions (R 0.627 to 0.936); in A3706-200313 (86 HD cells) and A3730-210323b (47) it does not
(R 0.114 and 0.328), and the first threshold crossing after DU comes at a median of 35 to 75 ms per
session, near the 35 ms the Methods give as the mode.